# Processing Sentinel-1 TOPS InSAR stack with ISCE3 & COMPASS
<br>  

**Author:** Zhenli Tang, Zhang Yunjun, August 3-7, 2026 [EarthScope InSAR Short Course (ISCE+)](https://www.earthscope.org/event/2026-technical-course-insar-processing-and-analysis-isce/).

---

## 0. Initial setup

Set up the Python environment, project paths, SARForge toolchain, and define
the area of interest (AOI). All common imports are consolidated here to avoid
redundancy in downstream cells.


In [ ]:
# === Standard library ===
import gc, os, re, glob, stat, time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlretrieve

# === Third-party ===
import numpy as np
import yaml
from compass.utils.iono import download_ionex
from matplotlib import pyplot as plt
from osgeo import gdal
plt.rcParams.update({'font.size': 12})

# === Local ===
import utils as ut

# ---------------------------------------------------------------------------
# Configuration -- dataset time/space info
# ---------------------------------------------------------------------------
start_date = '2024-07-01'
end_date   = '2024-10-11'
wsen  = (-155.50, 19.30, -154.95, 19.55)

# ---------------------------------------------------------------------------
# Configuration -- data structure
# ---------------------------------------------------------------------------
work_dir = Path('~/data/Hawaii_S1_A124').expanduser()
work_dir.mkdir(parents=True, exist_ok=True)
os.chdir(work_dir)
print('Go to directory:', work_dir)

burst_db_path = work_dir.parent / 's1-burst-db' / 'opera-burst-bbox-only.sqlite3'
slc_dir = work_dir / 'SLC'
orbit_dir = work_dir / 'orbits'
dem_path = work_dir / 'DEM' / 'cop_dem.tif'
tec_dir = work_dir / 'TEC'                # global ionospheric maps files
cslc_dir = work_dir / 'CSLC'              # coregistered SLC files
ifgram_dir = work_dir / 'interferograms'  # inteferograms

for d in [slc_dir, orbit_dir, tec_dir, cslc_dir, ifgram_dir]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Configuration -- processing parameters
# ---------------------------------------------------------------------------
rglks = 4               # number of looks in range direction
azlks = 2               # number of looks in azimuth direction
filt_strength = 0.5     # Goldstein filter strength

# ---------------------------------------------------------------------------
# Start-up
# ---------------------------------------------------------------------------
gc.collect()

## 1. Download SAR and auxliary data

This section prepares the data environment: burst database, SLC downloads,
orbit files, and DEM. These are prerequisites for the ISCE3 CSLC generation
and the subsequent SARForge InSAR pipeline.


### 1.0 Prepare OPERA Sentinel-1 burst ID database

Download and prepare the OPERA burst database required for Sentinel-1 burst
identification. The database (`opera-burst-bbox-only.sqlite3`) maps burst
IDs to geographic bounding boxes, enabling spatial queries for CSLC generation.


In [ ]:
# Download the pre-built OPERA burst ID database from GitHub
if not burst_db_path.exists():
    print('Downloading pre-built OPERA burst ID database from GitHub...')
    os.makedirs(os.path.dirname(burst_db_path), exist_ok=True)
    BURST_DB_URL = 'https://github.com/opera-adt/burst_db/releases/download/v0.10.0/opera-burst-bbox-only.sqlite3'
    urlretrieve(BURST_DB_URL, burst_db_path)
    print('OPERA burst ID database downloaded successfully!')
else:
    print(f'OPERA burst ID database already exist at {burst_db_path}.')

### 1.1 SLC Data Download

Download Sentinel-1 SLC products for the target area and time range using
`burst2stack`. This queries ASF/ESA for available scenes matching the AOI,
relative orbit, and swath specification.


In [ ]:
ext_str = " ".join([str(x) for x in wsen])
!burst2stack --rel-orbit 124 --all-anns --pols VV --swaths IW2 --output-dir {slc_dir} --start-date {start_date} --end-date {end_date} --extent {ext_str}


### 1.2 Orbit Files

Download precise orbit files (EOF) for each SLC scene using `eof`. Accurate
orbits are essential for geocoding and interferogram formation.


In [ ]:
!eof --search-path {slc_dir} --save-dir {orbit_dir} --force-asf

### 1.3 DEM Preparation

Download a digital elevation model (DEM) covering the AOI using `sardem`.
The DEM provides topographic phase removal during CSLC generation. A water-
body mask can be optionally downloaded for quality control.


In [ ]:
# use a bounding box much larger than the specified AOI above
# to cover all downloaded burst SLCs for the intermediate processing
buf = 2  # degree
dem_wsen = (np.floor(wsen[0] - buf), np.floor(wsen[1] - buf/2),
            np.ceil(wsen[2] + buf), np.ceil(wsen[3] + buf/2))
dem_path.parent.mkdir(parents=True, exist_ok=True)

# download Copernicus DEM
dem_wsen_str = " ".join([str(x) for x in dem_wsen])
!sardem --bbox {dem_wsen_str} --output-type float32 --output-format GTiff --data-source COP -o {dem_path}

# download NASADEM water-body mask
ut.download_nasadem_water_mask(dem_wsen, dem_path.parent)


### 1.4 IONEX TEC Download

Download IONEX Total Electron Content (TEC) files for ionospheric
phase correction using COMPASS's `download_ionex()`.  Each SLC date
gets its own daily TEC map (JPL final solution, 2-hour intervals).

Requires [NASA Earthdata Login](https://urs.earthdata.nasa.gov)
configured in `~/.netrc`:

```
machine urs.earthdata.nasa.gov login <user> password <pass>
```


In [ ]:
# discover all acquisition dates from downloaded SAFE files
date_list = ut.get_date_list(slc_dir)
print(f'Acquisition dates found ({len(date_list)}): {date_list}')

# download TEC file for each date
for i, date_str in enumerate(date_list):
    tec_file = download_ionex(date_str, str(tec_dir), sol_code='jpl')
    print(f'  {date_str}: {Path(tec_file).name}')

print('IONEX download complete.')


## 2. CSLC Generation


### 2.1 Generate config/run files for stack coregistration via `s1_geocode_stack`


In [ ]:
ext_str = " ".join([str(x) for x in wsen])
!s1_geocode_stack.py -s {slc_dir} -d {dem_path} -o {orbit_dir} -w {cslc_dir} -dx 10 -dy 20 --common-bursts-only --burst-db-file {burst_db_path} --unzipped --bbox {ext_str}


In [ ]:
!echo "========== run_files (first .sh) =========="; \
cat "$(ls {cslc_dir}/run_files/*.sh | head -1)"; 
!echo "========== runconfigs (first .yaml) =========="; cat "$(ls {cslc_dir}/runconfigs/*.yaml | head -1)"

### 2.2 Update config files for the ionospheric correction

We post-process the generated runconfig YAML files to add
the `tec_file` entry so that `s1_geocode_slc.py` applies ionospheric
correction during geocoding.


In [ ]:
# ---- Inject TEC file paths into generated runconfigs ----
config_dir = cslc_dir / 'runconfigs'

# 1. Build date -> tec_file mapping from downloaded IONEX files
tec_map = {}
for f in sorted(tec_dir.glob('*GIM.INX')):
    # Long IGS product name: IGS0OPSRAP_YYYYDDD0000_01D_02H_GIM.INX
    m = re.search(r'_(\d{4})(\d{3})\d{4}_', f.name)
    if m:
        y, doy = int(m.group(1)), int(m.group(2))
        dt = datetime(y, 1, 1) + timedelta(days=doy - 1)
        tec_map[dt.strftime('%Y%m%d')] = str(f)
        continue

for f in sorted(tec_dir.glob('jplg*.*i')):
    # Legacy name: jplgDDD0.YYi
    m = re.search(r'jplg(\d{3})0\.(\d{2})i', f.name)
    if m:
        doy, yy = int(m.group(1)), int(m.group(2))
        y = 2000 + yy
        dt = datetime(y, 1, 1) + timedelta(days=doy - 1)
        tec_map[dt.strftime('%Y%m%d')] = str(f)
print(f'IONEX date map: {len(tec_map)} files')

# 2. update config YAML file
updated = 0
for cfg_path in sorted(config_dir.glob('geo_runconfig_????????_*.yaml')):
    # Expect name format: geo_runconfig_YYYYMMDD_tRRR_BBBBBB_iwN.yaml
    parts = cfg_path.stem.split('_')
    if len(parts) < 4:
        continue
    date_str = parts[2]  # YYYYMMDD
    if date_str not in tec_map:
        continue
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    groups = cfg['runconfig']['groups']
    if 'tec_file' not in groups['dynamic_ancillary_file_group'] or groups['dynamic_ancillary_file_group']['tec_file'] == None:
        groups['dynamic_ancillary_file_group']['tec_file'] = tec_map[date_str]
        with open(cfg_path, 'w') as f:
            yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
        updated += 1
total = len(list(config_dir.glob('geo_runconfig_*.yaml')))
print(f'Runconfigs updated with tec_file: {updated} / {total}')


### 2.3 Run stack coregistration

Execute the per-burst shell scripts generated by `s1_geocode_stack.py` with
bounded concurrency (`MAX_CONCURRENT = 4`). Completed outputs are detected and
skipped on re-run, supporting resumption after interruption.

Output: `./CSLC/{burst_id}/{date}/{burst_id}_{date}.h5`


In [ ]:
# grab all run_files
run_dir = cslc_dir / 'run_files'
run_files = [str(x) for x in run_dir.glob('run_*.sh')]
# change file permission to executables
for run_file in run_files:
    current_permissions = os.stat(run_file).st_mode
    new_permissions = current_permissions | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH
    os.chmod(run_file, new_permissions)

# run
for i, run_file in enumerate(run_files):
    print('-'*20 + f'generating the {i+1}/{len(run_files)} GSLC burst...')

    # grab output CSLC file path
    prefix = os.path.splitext(os.path.basename(run_file))[0]
    date_str = prefix.split('_')[1]
    burst_id = prefix.split(date_str)[1][1:]
    cslc_path = cslc_dir / burst_id / date_str / f'{burst_id}_{date_str}.h5'

    # skip re-generating if exists
    if os.path.exists(cslc_path):
        print(f'CSLC file exists at: {cslc_path}, skip re-generating.')
    else:
        !{run_file}


### 2.4 Download static layers

In [ ]:
burst_id_list = sorted(os.path.basename(d) for d in cslc_dir.iterdir() if d.is_dir() and d.name.startswith('t'))
date_list = ut.get_date_list(slc_dir)
downloaded = ut.download_opera_static_layers(
    burst_id_list, work_dir, date_list[0], bbox_wsen=wsen,
)

### 2.5 Compute baselines time series

In [ ]:
print(f'Burst IDs: {burst_id_list}')

ut.compute_baselines_for_bursts(
    burst_ids=burst_id_list,
    cslc_dir = cslc_dir,
    output_base=str(work_dir / 'baselines' / 'bursts')
)

# average multi burst baselines into single baseline
ut.merge_baselines(
    baseline_dir=str(work_dir / 'baselines' / 'bursts'),
    output_dir=str(work_dir / 'baselines'),
)

## 3. Interferogram stack generation

The pipeline converts coregistered burst CSLCs into unwrapped interferograms
ready for time-series analysis (e.g., MintPy), in four steps:

1. **Select interferometric pairs** — Select sequential network via number of connections
2. **Generate multilooked stitched interferograms**
3. **Filtering & phase-sigma coherence estimation**
4. **Phase unwrapping**

### 3.1 Select interferometric pairs

Scan CSLC files across all bursts and generate a sequential network of
interferometric pairs via the number of connections. All bursts share the
same pair list since acquisition dates are aligned.

- **Input**: `./CSLC/` (all burst subdirectories)
- **Output**: `./process/ifgrams/date12_list.txt`
- **Key option**: `-n N` — maximum temporal separation between pairs


In [ ]:
date12_list = ut.generate_ifgram_pairs(str(cslc_dir), str(ifgram_dir), n_connections=2)


### 3.2 Generate (stitched) interferograms

Form stitched, multilooked interferograms and complex coherence from burst
CSLCs with a single sequential call:

1. **Per burst** — read only the AOI-cropped part (a fraction of the burst
   in memory; the entire burst when `bbox_wsen=None`), align reference &
   secondary to a common grid, form the interferogram (and complex
   coherence if `save_full_res=True`), keep only the small cropped pieces.
2. **Stitch** — combine all per-burst pieces into a single stitched
   interferogram (and coherence) via pixel-offset copying.
3. **Multilook** — average the stitched interferogram in azimuth/range and
   write the multilooked GeoTIFF. Full-resolution stitched ifg/coh are
   saved only when `save_full_res=True`.

- **Input**: `./CSLC/{burst_id}/{date}/{burst_id}_{date}.h5` + pair list
- **Output** `./interferograms/{d1}_{d2}/mli.int.tif` — multilooked interferogram


In [ ]:
# read the pair list generated in 3.1
date12_list = list(np.loadtxt(ifgram_dir / 'date12_list.txt', dtype=bytes).astype(str))
num_pair = len(date12_list)
print(f'{num_pair} pairs')

# generate stitched multilooked ifg (sequential, no parallel)
# save_full_res=False: skip complex coherence + full-res ifg/coh files
# bbox_wsen=None -> process the ENTIRE burst(s): each burst is read in full
#   and stitched at its union extent; the buffer argument is ignored.
# bbox_wsen=wsen -> crop each burst around the AOI expanded by buffer (default).
ifg_ml_list, coh_list = ut.generate_stitched_ifgrams(
    cslc_dir      = cslc_dir,
    date12_list   = date12_list,
    output_dir    = ifgram_dir,
    bbox_wsen     = wsen,       # None -> entire burst(s); AOI wsen + buffer otherwise
    buffer        = 0.05,       # buffer (deg) around the AOI, ignored when bbox_wsen=None
    coh_win       = 5,
    lks_y         = azlks,      # azimuth looks
    lks_x         = rglks,      # range looks
    save_full_res = False,      # optional: save full-res stitched ifg/coh
    save_cropped_slc = False,   # optional: save per-burst cropped SLC GeoTIFFs
    save_ifgs        = False,   # optional: save per-burst ifg/coh GeoTIFFs
)
print(f'Complete {num_pair} multilooked stitched interferogram generation.')


In [ ]:
# demo for plot
date12 = date12_list[0]
pair_dir = ifgram_dir / date12
ifg_ml_file = sorted(pair_dir.glob('mli.int.tif'))[0]

fig, ax = plt.subplots(figsize=(8, 4))
ds = gdal.Open(str(ifg_ml_file))
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None
im = ax.imshow(np.angle(arr), aspect='auto', cmap='RdBu_r', vmin=-np.pi, vmax=np.pi)
ax.set_title(f'Multilooked Wrapped Phase ({date12})', fontsize=12)
ax.set_xlabel('Range (pixels)')
ax.set_ylabel('Azimuth (pixels)')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Phase (rad)')
plt.tight_layout()
plt.show()


### 3.3 Filtering & phase-sigma estimation

Apply adaptive Goldstein phase filtering to reduce interferometric noise,
then estimate the phase-sigma coherence as a quality metric for unwrapping.

- **Input**: `./interferograms/{d1}_{d2}/mli.int.tif`
- **Output** `./interferograms/{d1}_{d2}`:
  - `filt_mli.int.tif`
  - `filt_mli.phsig.coh.tif`
- **Key options**: `--alpha` (default 0.5), `--ps-window-size`


In [ ]:
# process each pair directory in place
for i, date12 in enumerate(date12_list):
    print('-'*25 + f'{i+1}/{num_pair}' + '-'*25)
    in_file = ifgram_dir / date12 / 'mli.int.tif'
    out_file = ifgram_dir / date12 / 'filt_mli.int.tif'
    ut.filter_tif(in_file, out_file, alpha=filt_strength)
    ut.generate_phsig_coh_tif(out_file)
print(f'Complete filtering & phase-sigma estimation for {num_pair} pairs.')


In [ ]:
# demo for plot
date12 = date12_list[0]
ifg_file = ifgram_dir / date12 / 'filt_mli.int.tif'
coh_file = ifgram_dir / date12 / 'filt_mli.phsig.coh.tif'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

ds = gdal.Open(str(ifg_file))
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None
im1 = ax1.imshow(np.angle(arr), aspect='auto', cmap='RdBu_r', vmin=-np.pi, vmax=np.pi)
ax1.set_title(f'Filtered Wrapped Phase ({date12})', fontsize=12)
ax1.set_xlabel('Range (pixels)'); ax1.set_ylabel('Azimuth (pixels)')
cbar1 = plt.colorbar(im1, ax=ax1, shrink=0.85); cbar1.set_label('Phase (rad)')

ds = gdal.Open(str(coh_file))
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None
im2 = ax2.imshow(arr, aspect='auto', cmap='gray', vmin=0, vmax=1)
ax2.set_title(f'Phase-Sigma Coherence ({date12})', fontsize=12)
ax2.set_xlabel('Range (pixels)'); ax2.set_ylabel('Azimuth (pixels)')
cbar2 = plt.colorbar(im2, ax=ax2, shrink=0.85); cbar2.set_label('Coherence')

plt.tight_layout()
plt.show()


### 3.4 Phase unwrapping

Unwrap filtered interferograms using SNAPHU, guided by the phase-sigma
coherence map. The unwrapped phase is the final product, ready for
time-series analysis with MintPy.

- **Input**: `./interferograms/{d1}_{d2}/filt_mli.int.tif` + `./interferograms/{d1}_{d2}/filt_mli.phsig.coh.tif`
- **Output**: `./interferograms/{d1}_{d2}/filt_mli.unw.tif`


In [ ]:
water_mask_path = dem_path.parent / 'swbd_nasadem.wbd'
ncorrlooks = azlks * rglks / (1.2**2)

for i, date12 in enumerate(date12_list):
    print('-'*20 + f'{i+1}/{num_pair}')
    ifg_file = ifgram_dir / date12 / 'filt_mli.int.tif'
    coh_file = ifgram_dir / date12 / 'filt_mli.phsig.coh.tif'
    unw_file = ifgram_dir / date12 / 'filt_mli.unw.tif'
    ut.unwrap_single_ifgram(
        ifg_file, coh_file, unw_file, water_mask=water_mask_path,
        nlooks=ncorrlooks, cost_mode='smooth', init_method='mcf',
    )
print(f'Complete phase unwrapping for {num_pair} pairs.')


In [ ]:
# demo for plot
date12 = date12_list[0]
unw_file = ifgram_dir / date12 / 'filt_mli.unw.tif'

ds = gdal.Open(str(unw_file))
arr = ds.GetRasterBand(1).ReadAsArray()
ds = None

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(arr, aspect='auto', cmap='jet')
ax.set_title(f'Unwrapped Phase ({date12})', fontsize=12)
ax.set_xlabel('Range (pixels)')
ax.set_ylabel('Azimuth (pixels)')
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Phase (rad)')
plt.tight_layout()
plt.show()
